# Code 10.4.3: SARSA and Q-learning on CliffWalking

Tabular SARSA and Q-learning with ε-greedy exploration on Gymnasium's `CliffWalking-v1`. The two methods share everything except the target. `greedy_path` follows the learned greedy policy from the start state (for at most 100 steps) and reports its return.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# From Code 10.4.1
import numpy as np


In [ ]:
import gymnasium as gym

def eps_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(Q.shape[1]))
    q = Q[s]
    return int(rng.choice(np.flatnonzero(q == q.max())))     # random tie-break

def train(method, n_episodes=500, alpha=0.5, gamma=1.0, eps=0.1, seed=0):
    """Tabular SARSA or Q-learning; returns Q and the reward of every episode."""
    env = gym.make("CliffWalking-v1")
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    episode_rewards = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=seed * 10_000 + ep)
        a = eps_greedy(Q, s, eps, rng)
        total, done = 0.0, False
        while not done:
            s2, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated
            a2 = eps_greedy(Q, s2, eps, rng)
            if method == "sarsa":        # on-policy: the action we will really take
                target = r + gamma * Q[s2, a2] * (not terminated)
            else:                        # Q-learning: the greedy action
                target = r + gamma * Q[s2].max() * (not terminated)
            Q[s, a] += alpha * (target - Q[s, a])
            s, a = s2, a2
            total += r
        episode_rewards.append(total)
    return Q, np.array(episode_rewards)

def greedy_path(Q, max_steps=100):
    """Follow the greedy policy from the start; return visited states and return."""
    env = gym.make("CliffWalking-v1")
    s, _ = env.reset(seed=0)
    path, total = [s], 0.0
    for _ in range(max_steps):
        s, r, terminated, truncated, _ = env.step(int(Q[s].argmax()))
        path.append(s)
        total += r
        if terminated:
            break
    return path, total

for method in ("sarsa", "qlearning"):
    runs = [train(method, seed=i) for i in range(50)]
    online = np.mean([rewards[-100:].mean() for _, rewards in runs])
    greedy = [greedy_path(Q)[1] for Q, _ in runs]
    vals, counts = np.unique(greedy, return_counts=True)
    print(f"{method:9s} reward per episode (last 100): {online:6.1f}   greedy returns: "
          + ", ".join(f"{v:.0f} (x{c})" for v, c in zip(vals, counts)))


**Expected output**

Output:

```text
sarsa     reward per episode (last 100):  -26.3   greedy returns: -100 (x7), -19 (x3), -17 (x40)
qlearning reward per episode (last 100):  -49.3   greedy returns: -13 (x50)
```
